# Microsoft Agent Framework — Azure OpenAI (Responses API)

U ovom primjeru koda koristit ćete **Microsoft Agent Framework (MAF)** za izradu jednostavnog agenta kojeg pokreće **Azure OpenAI** koristeći **Responses API**.

> **Napomena o migraciji:** Ovaj se primjer prethodno koristio Semantic Kernel s GitHub modelima. Migriran je na Microsoft Agent Framework, a GitHub modeli (zastarjeli, penzioniraju se u srpnju 2026.) zamijenjeni su s Azure OpenAI, koji podržava Responses API. `OpenAIChatClient` u MAF-u cilja stabilnu Azure OpenAI `/openai/v1/` krajnju točku i prema zadanim postavkama koristi Responses API.

Svrha ovog primjera je demonstrirati korake koji će kasnije biti primijenjeni u dodatnim primjerima koda pri implementaciji različitih agenatskih obrazaca.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Uvezite potrebne Python pakete


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definiranje alata

U Microsoft Agent okviru, **alat** je obična Python funkcija ukrašena s `@tool` koju agent može pozvati. Ispod definiramo alat koji vraća slučajnu destinaciju za odmor i izbjegava ponavljanje prethodne.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Kreiranje Agenta

Ovdje stvaramo Agenta pod nazivom `TravelAgent`.

U ovom primjeru koristimo vrlo osnovne upute. Slobodno modificirajte ove upute kako biste promatrali kako se ponašanje agenta mijenja.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Pokretanje agenta

Sada možemo pokrenuti agenta. Kreiramo `AgentSession` kako bi agent zapamtio razgovor kroz više koraka, zatim pošaljemo dva `user_inputs`. Prvi traži putovanje; drugi kaže da korisniku nije bilo drago prijedloga i traži drugi — agent koristi povijest sesije plus alat `get_random_destination` za odgovor.

Možete prilagoditi ove poruke da biste vidjeli kako agent drugačije reagira. Odgovori se **emitiraju** token po token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
